# Week 2 — Tiled, register-blocked, vectorized SGEMM

Runtime → Change runtime type → **T4 GPU**. Skip naive (slow). Paste all three tables back into chat.


In [ ]:
import shutil, subprocess, sys
if shutil.which("nvidia-smi") is None:
    sys.exit("No GPU. Runtime → Change runtime type → T4 GPU.")
print(subprocess.check_output(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], text=True))

In [ ]:
%%bash
set -euo pipefail
if [ -d cuda-kernel-optimization ]; then
  cd cuda-kernel-optimization && git pull
else
  git clone https://github.com/preethamdandu/cuda-kernel-optimization.git && cd cuda-kernel-optimization
fi
ARCH=sm_$(nvidia-smi --query-gpu=compute_cap --format=csv,noheader | tr -d '.')
echo "building for $ARCH"
nvcc -O3 -arch=$ARCH -lineinfo benchmark/bench.cu src/*.cu -lcublas -o bench
./bench --stage tiled --sizes 1024 2048 4096
./bench --stage register --sizes 1024 2048 4096
./bench --stage vectorized --sizes 1024 2048 4096